# 00.6 Query Pose To Replacement Candidates

Upload a single dog image, inspect the detected pose with visualization, and automatically retrieve the 50 best replacement-dog candidates from the current replaceable pool.


In [ ]:
%matplotlib inline


## What This Module Produces

- interactive image upload
- query pose visualization
- replacement-oriented pose summary
- top-50 replacement candidates displayed directly in the notebook
- per-candidate download buttons
- saved JSON and image outputs under `data/outputs/replacement_candidate_retrieval/`


In [ ]:
import base64
import io
import json
import math
import os
import shutil
from datetime import datetime
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from IPython.display import HTML, display, clear_output
from PIL import Image
from ultralytics import YOLO


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current notebook path.")


NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = find_repo_root(NOTEBOOK_DIR)
DATA_ROOT = ROOT / "data" / "Images"
POSE_MODEL_PATH = Path(
    os.environ.get("DOG_POSE_MODEL_PATH", str(ROOT / "models" / "dog_pose_best.pt"))
).resolve()
CURRENT_STAGE_INDEX_PATH = ROOT / "data" / "outputs" / "pose_index" / "current_stage_side_profiles" / "current_stage_side_profiles.json"
POSE_INDEX_PATH = ROOT / "data" / "outputs" / "pose_index" / "dog_pose_index.json"
OUTPUT_ROOT = ROOT / "data" / "outputs" / "replacement_candidate_retrieval"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

POSE_CONF = 0.15
POSE_IMGSZ = 960
KEYPOINT_CONF = 0.20
TOP_K = 50

DEVICE = 0 if torch.cuda.is_available() else None
if DEVICE is None:
    raise RuntimeError("This notebook is local-GPU only. CUDA was not detected.")
if not POSE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Dog pose model not found: {POSE_MODEL_PATH}")
if not CURRENT_STAGE_INDEX_PATH.exists():
    raise FileNotFoundError(f"Current-stage pose index not found: {CURRENT_STAGE_INDEX_PATH}")

print("Project root:", ROOT)
print("Pose model:", POSE_MODEL_PATH)
print("Current-stage index:", CURRENT_STAGE_INDEX_PATH)
print("CUDA device:", torch.cuda.get_device_name(0))


## Pose Labels And Skeleton Definitions


In [ ]:
DOG_KEYPOINT_NAMES = [
    "front_left_paw",
    "front_left_knee",
    "front_left_elbow",
    "rear_left_paw",
    "rear_left_knee",
    "rear_left_elbow",
    "front_right_paw",
    "front_right_knee",
    "front_right_elbow",
    "rear_right_paw",
    "rear_right_knee",
    "rear_right_elbow",
    "tail_start",
    "tail_end",
    "left_ear_base",
    "right_ear_base",
    "nose",
    "chin",
    "left_ear_tip",
    "right_ear_tip",
    "left_eye",
    "right_eye",
    "withers",
    "throat",
]
NAME_TO_INDEX = {name: idx for idx, name in enumerate(DOG_KEYPOINT_NAMES)}

PART_GROUPS = {
    "head": ["nose", "chin", "left_eye", "right_eye", "left_ear_base", "right_ear_base", "left_ear_tip", "right_ear_tip"],
    "torso": ["withers", "throat", "tail_start"],
    "front_legs": ["front_left_paw", "front_left_knee", "front_left_elbow", "front_right_paw", "front_right_knee", "front_right_elbow"],
    "hind_legs": ["rear_left_paw", "rear_left_knee", "rear_left_elbow", "rear_right_paw", "rear_right_knee", "rear_right_elbow"],
    "tail": ["tail_start", "tail_end"],
}

POSE_EDGES = [
    ("nose", "chin"),
    ("nose", "left_eye"),
    ("nose", "right_eye"),
    ("left_eye", "left_ear_base"),
    ("right_eye", "right_ear_base"),
    ("left_ear_base", "left_ear_tip"),
    ("right_ear_base", "right_ear_tip"),
    ("throat", "chin"),
    ("withers", "throat"),
    ("withers", "tail_start"),
    ("tail_start", "tail_end"),
    ("withers", "front_left_elbow"),
    ("front_left_elbow", "front_left_knee"),
    ("front_left_knee", "front_left_paw"),
    ("withers", "front_right_elbow"),
    ("front_right_elbow", "front_right_knee"),
    ("front_right_knee", "front_right_paw"),
    ("tail_start", "rear_left_elbow"),
    ("rear_left_elbow", "rear_left_knee"),
    ("rear_left_knee", "rear_left_paw"),
    ("tail_start", "rear_right_elbow"),
    ("rear_right_elbow", "rear_right_knee"),
    ("rear_right_knee", "rear_right_paw"),
]


## Shared Helper Functions


In [ ]:
def read_rgb(path: Path) -> np.ndarray:
    arr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if arr is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)


def clip_bbox(xyxy, width, height):
    x1, y1, x2, y2 = [float(v) for v in xyxy]
    x1 = max(0.0, min(x1, width - 1))
    y1 = max(0.0, min(y1, height - 1))
    x2 = max(x1 + 1.0, min(x2, width))
    y2 = max(y1 + 1.0, min(y2, height))
    return [x1, y1, x2, y2]


def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def bbox_width_height(bbox):
    x1, y1, x2, y2 = bbox
    return max(1e-6, x2 - x1), max(1e-6, y2 - y1)


def is_valid_point(point):
    return (
        point is not None
        and len(point) == 2
        and point[0] is not None
        and point[1] is not None
        and not (np.isnan(point[0]) or np.isnan(point[1]))
    )


def record_point(record, name, field="keypoints_xy"):
    idx = NAME_TO_INDEX[name]
    points = record.get(field) or []
    if idx >= len(points):
        return None
    point = points[idx]
    if not is_valid_point(point):
        return None
    return np.array(point, dtype=np.float32)


def point_from_arrays(points, confs, name, threshold=KEYPOINT_CONF):
    idx = NAME_TO_INDEX[name]
    if idx >= len(points):
        return None
    point = points[idx]
    if point is None:
        return None
    point = np.array(point, dtype=np.float32)
    if point.shape != (2,) or np.isnan(point).any():
        return None
    conf = float(confs[idx]) if confs is not None and idx < len(confs) else 1.0
    if conf < threshold:
        return None
    return point


def orientation_label(points, confs):
    nose = point_from_arrays(points, confs, "nose")
    tail_start = point_from_arrays(points, confs, "tail_start")
    if nose is None or tail_start is None:
        return "unknown"
    return "left" if nose[0] < tail_start[0] else "right"


def frame_fit_bucket(bbox, width, height, margin=4):
    x1, y1, x2, y2 = bbox
    touch_count = int(x1 <= margin) + int(y1 <= margin) + int(x2 >= width - margin) + int(y2 >= height - margin)
    if touch_count > 0:
        return "truncated_by_frame", touch_count
    return "in_frame", touch_count


def visible_part_signature(points, confs):
    visible_parts = []
    rules = {
        "head": 2,
        "torso": 1,
        "front_legs": 2,
        "hind_legs": 2,
        "tail": 1,
    }
    for part_name, point_names in PART_GROUPS.items():
        visible_count = sum(point_from_arrays(points, confs, name) is not None for name in point_names)
        if visible_count >= rules[part_name]:
            visible_parts.append(part_name)
    return "+".join(visible_parts) if visible_parts else "none"


def coverage_class(points, confs):
    signature = visible_part_signature(points, confs)
    required = {"head", "torso", "front_legs", "hind_legs", "tail"}
    present = set(signature.split("+")) if signature != "none" else set()
    return "full_body" if required.issubset(present) else "partial_body"


def body_axis(points, confs):
    nose = point_from_arrays(points, confs, "nose")
    tail_start = point_from_arrays(points, confs, "tail_start")
    if nose is None or tail_start is None:
        return None
    return nose - tail_start


def body_angle_deg(points, confs):
    axis = body_axis(points, confs)
    if axis is None:
        return None
    return float(np.degrees(np.arctan2(axis[1], axis[0])))


def body_angle_bucket(points, confs):
    angle = body_angle_deg(points, confs)
    if angle is None:
        return "unknown"
    abs_angle = abs(angle) % 180.0
    if abs_angle > 90.0:
        abs_angle = 180.0 - abs_angle
    if abs_angle <= 20:
        return "horizontal"
    if abs_angle <= 55:
        return "diagonal"
    return "vertical"


def posture_bucket(bbox):
    width, height = bbox_width_height(bbox)
    ratio = height / max(width, 1e-6)
    if ratio < 0.62:
        return "low"
    if ratio > 0.92:
        return "tall"
    return "mid"


def spread_bucket(points, confs, bbox):
    paws = [
        point_from_arrays(points, confs, "front_left_paw"),
        point_from_arrays(points, confs, "front_right_paw"),
        point_from_arrays(points, confs, "rear_left_paw"),
        point_from_arrays(points, confs, "rear_right_paw"),
    ]
    paws = [pt for pt in paws if pt is not None]
    width, _ = bbox_width_height(bbox)
    if len(paws) < 2:
        return "unknown"
    xs = np.array([pt[0] for pt in paws], dtype=np.float32)
    normalized_span = float((xs.max() - xs.min()) / max(width, 1e-6))
    if normalized_span >= 0.62:
        return "extended"
    if normalized_span >= 0.42:
        return "balanced"
    return "compact"


def classify_view(points, confs, bbox):
    orientation = orientation_label(points, confs)
    signature = visible_part_signature(points, confs)
    coverage = coverage_class(points, confs)
    angle_bucket = body_angle_bucket(points, confs)
    width, height = bbox_width_height(bbox)
    bbox_ratio = width / max(height, 1e-6)
    if orientation != "unknown" and coverage == "full_body" and "front_legs" in signature and "hind_legs" in signature and bbox_ratio > 1.0:
        return "side_profile"
    if orientation == "unknown" and angle_bucket == "vertical":
        return "frontal_or_rear"
    return "ambiguous"


def canonical_normalized_keypoints(points, confs, bbox):
    valid_points = [point_from_arrays(points, confs, name, threshold=-1.0) for name in DOG_KEYPOINT_NAMES]
    valid_points = [pt for pt in valid_points if pt is not None]
    if not valid_points:
        return [[None, None] for _ in DOG_KEYPOINT_NAMES]

    tail_start = point_from_arrays(points, confs, "tail_start")
    withers = point_from_arrays(points, confs, "withers")
    nose = point_from_arrays(points, confs, "nose")

    if tail_start is not None and withers is not None:
        center = 0.5 * (tail_start + withers)
    else:
        center = np.mean(np.stack(valid_points), axis=0)

    if tail_start is not None and nose is not None:
        scale = float(np.linalg.norm(nose - tail_start))
    else:
        box_w, box_h = bbox_width_height(bbox)
        scale = max(box_w, box_h)
    scale = max(scale, 1e-6)

    orientation = orientation_label(points, confs)
    normalized = []
    for name in DOG_KEYPOINT_NAMES:
        pt = point_from_arrays(points, confs, name, threshold=-1.0)
        if pt is None:
            normalized.append([None, None])
            continue
        value = (pt - center) / scale
        if orientation == "right":
            value = np.array([-value[0], value[1]], dtype=np.float32)
        normalized.append([float(value[0]), float(value[1])])
    return normalized


def shape_vector_from_normalized(record):
    def norm_point(name):
        return record_point(record, name, field="normalized_keypoints")

    xs = []
    ys = []
    for point in record.get("normalized_keypoints") or []:
        if is_valid_point(point):
            xs.append(point[0])
            ys.append(point[1])
    if not xs:
        return [None] * 8

    width = float(max(xs) - min(xs))
    height = float(max(ys) - min(ys))
    aspect = width / max(height, 1e-6)

    def segment_length(a_name, b_name):
        a = norm_point(a_name)
        b = norm_point(b_name)
        if a is None or b is None:
            return None
        return float(np.linalg.norm(a - b))

    def mean_optional(values):
        values = [float(v) for v in values if v is not None and not np.isnan(v)]
        if not values:
            return None
        return float(np.mean(values))

    body_len = segment_length("nose", "tail_start")
    front_leg = mean_optional([
        segment_length("front_left_elbow", "front_left_paw"),
        segment_length("front_right_elbow", "front_right_paw"),
    ])
    hind_leg = mean_optional([
        segment_length("rear_left_elbow", "rear_left_paw"),
        segment_length("rear_right_elbow", "rear_right_paw"),
    ])
    head_len = mean_optional([
        segment_length("nose", "chin"),
        segment_length("nose", "left_ear_base"),
        segment_length("nose", "right_ear_base"),
    ])
    tail_len = segment_length("tail_start", "tail_end")

    return [
        width,
        height,
        aspect,
        float(body_len) if body_len is not None else None,
        float(front_leg) if front_leg is not None else None,
        float(hind_leg) if hind_leg is not None else None,
        float(head_len) if head_len is not None else None,
        float(tail_len) if tail_len is not None else None,
    ]


def shape_label_from_vector(vector):
    aspect = vector[2]
    front_leg = vector[4]
    hind_leg = vector[5]
    if aspect is None:
        return "unknown"
    leg_values = [float(v) for v in [front_leg, hind_leg] if v is not None and not np.isnan(v)]
    mean_leg = float(np.mean(leg_values)) if leg_values else None
    if aspect >= 2.15 and (mean_leg is None or mean_leg >= 0.55):
        return "slender"
    if aspect <= 1.35:
        return "compact"
    if mean_leg is not None and mean_leg >= 0.72:
        return "long_legged"
    return "balanced"


def current_stage_gate(record):
    signature = record["visible_parts_signature"]
    required_parts = {"head", "torso", "front_legs", "hind_legs", "tail"}
    visible_parts = set(signature.split("+")) if signature != "none" else set()
    if record["status"] != "ok":
        return False, "no_pose"
    if record["dog_instance_count"] > 1:
        return False, "multiple_dogs_present"
    if record["view_class"] != "side_profile":
        return False, "not_side_profile"
    if record["coverage_class"] != "full_body":
        return False, "not_full_body"
    if record["frame_fit_bucket"] != "in_frame":
        return False, "truncated_by_frame"
    if record["visible_keypoints"] < 12:
        return False, "too_few_keypoints"
    if not required_parts.issubset(visible_parts):
        return False, "missing_required_parts"
    if record["bbox_area_ratio"] > 0.55:
        return False, "dog_too_large_in_frame"
    return True, "pass"


def query_pose_record(image_path: Path, pose_model):
    image = read_rgb(image_path)
    height, width = image.shape[:2]
    result = pose_model.predict(source=image, conf=POSE_CONF, imgsz=POSE_IMGSZ, device=DEVICE, verbose=False)[0]

    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return {
            "status": "no_pose",
            "query_image_path": str(image_path),
            "width": width,
            "height": height,
            "dog_instance_count": 0,
        }

    keypoints_xy_all = result.keypoints.xy.detach().cpu().numpy() if result.keypoints is not None else None
    keypoints_conf_all = result.keypoints.conf.detach().cpu().numpy() if result.keypoints is not None else None

    candidates = []
    for idx, box in enumerate(boxes):
        xyxy = clip_bbox(box.xyxy[0].detach().cpu().numpy().tolist(), width, height)
        score = float(box.conf.item())
        area = bbox_area(xyxy)
        keypoints_xy = keypoints_xy_all[idx].tolist() if keypoints_xy_all is not None else [[None, None] for _ in DOG_KEYPOINT_NAMES]
        keypoints_conf = keypoints_conf_all[idx].tolist() if keypoints_conf_all is not None else [0.0] * len(DOG_KEYPOINT_NAMES)
        candidates.append({
            "bbox_xyxy": xyxy,
            "detection_conf": score,
            "bbox_area": area,
            "keypoints_xy": keypoints_xy,
            "keypoints_conf": keypoints_conf,
        })

    dog_instance_count = len(candidates)
    best = max(candidates, key=lambda item: item["detection_conf"] * math.sqrt(max(item["bbox_area"], 1e-6)))
    points = best["keypoints_xy"]
    confs = best["keypoints_conf"]
    visible_names = [name for name, conf in zip(DOG_KEYPOINT_NAMES, confs) if conf >= KEYPOINT_CONF]
    frame_bucket, edge_touch_count = frame_fit_bucket(best["bbox_xyxy"], width, height)
    geometry_label = "|".join([
        body_angle_bucket(points, confs),
        posture_bucket(best["bbox_xyxy"]),
        spread_bucket(points, confs, best["bbox_xyxy"]),
    ])

    record = {
        "status": "ok",
        "query_image_path": str(image_path),
        "width": width,
        "height": height,
        "dog_instance_count": dog_instance_count,
        "bbox_xyxy": best["bbox_xyxy"],
        "bbox_area_ratio": float(best["bbox_area"] / max(width * height, 1e-6)),
        "detection_conf": best["detection_conf"],
        "keypoints_xy": points,
        "keypoints_conf": confs,
        "visible_keypoints": len(visible_names),
        "visible_keypoint_names": visible_names,
        "visible_parts_signature": visible_part_signature(points, confs),
        "coverage_class": coverage_class(points, confs),
        "original_orientation": orientation_label(points, confs),
        "view_class": classify_view(points, confs, best["bbox_xyxy"]),
        "frame_fit_bucket": frame_bucket,
        "frame_edge_touch_count": edge_touch_count,
        "pose_geometry_label": geometry_label,
        "body_angle_deg": body_angle_deg(points, confs),
        "normalized_keypoints": canonical_normalized_keypoints(points, confs, best["bbox_xyxy"]),
    }
    record["shape_vector"] = shape_vector_from_normalized(record)
    record["shape_label"] = shape_label_from_vector(record["shape_vector"])
    record["current_pipeline_pass"], record["current_pipeline_reason"] = current_stage_gate(record)
    record["current_candidate_pool_key"] = "|".join([
        record["view_class"],
        record["pose_geometry_label"],
        record["coverage_class"],
        record["visible_parts_signature"],
    ])
    return record


def normalized_array_from_record(record):
    arr = []
    for point in record.get("normalized_keypoints") or []:
        if is_valid_point(point):
            arr.append([float(point[0]), float(point[1])])
        else:
            arr.append([np.nan, np.nan])
    return np.array(arr, dtype=np.float32)


def keypoint_distance(record_a, record_b, min_shared=10):
    a = normalized_array_from_record(record_a)
    b = normalized_array_from_record(record_b)
    shared = np.isfinite(a[:, 0]) & np.isfinite(a[:, 1]) & np.isfinite(b[:, 0]) & np.isfinite(b[:, 1])
    shared_count = int(shared.sum())
    if shared_count < min_shared:
        return math.inf, shared_count
    diff = a[shared] - b[shared]
    dist = float(np.sqrt((diff ** 2).sum(axis=1).mean()))
    return dist, shared_count


def visibility_overlap(record_a, record_b):
    vis_a = set(record_a.get("visible_keypoint_names") or [])
    vis_b = set(record_b.get("visible_keypoint_names") or [])
    if not vis_a and not vis_b:
        return 0.0
    return float(len(vis_a & vis_b)) / max(len(vis_a | vis_b), 1)


def shape_distance(record_a, record_b):
    q_vec = record_a.get("shape_vector") or []
    c_vec = record_b.get("shape_vector") or []
    diffs = []
    shared = 0
    for q_val, c_val in zip(q_vec, c_vec):
        if q_val is None or c_val is None:
            continue
        if np.isnan(q_val) or np.isnan(c_val):
            continue
        diffs.append(abs(float(q_val) - float(c_val)))
        shared += 1
    if shared == 0:
        return 1.0, 0
    return float(np.mean(diffs)), shared


def geometry_penalty(record_a, record_b):
    if record_a.get("current_candidate_pool_key") == record_b.get("current_candidate_pool_key"):
        return 0.0
    if record_a.get("pose_geometry_label") == record_b.get("pose_geometry_label"):
        return 0.04
    return 0.09


def draw_pose_record(record, image_path=None, figsize=(9, 7), title=None):
    image_path = Path(image_path or record["query_image_path"])
    image = read_rgb(image_path)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image)
    ax.axis("off")

    if record.get("status") != "ok":
        ax.set_title("No dog pose was detected.")
        return fig

    x1, y1, x2, y2 = record["bbox_xyxy"]
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2.5, edgecolor="#00C2A8", facecolor="none")
    ax.add_patch(rect)

    point_lookup = {name: record_point(record, name, field="keypoints_xy") for name in DOG_KEYPOINT_NAMES}
    for a_name, b_name in POSE_EDGES:
        a = point_lookup[a_name]
        b = point_lookup[b_name]
        if a is not None and b is not None:
            ax.plot([a[0], b[0]], [a[1], b[1]], color="#FFD166", linewidth=2.0)

    for point in point_lookup.values():
        if point is not None:
            ax.scatter(point[0], point[1], s=26, color="#EF476F")

    if title is None:
        title = (
            f"Pose | geometry={record['pose_geometry_label']} | "
            f"shape={record['shape_label']} | pass={record['current_pipeline_pass']}"
        )
    ax.set_title(title)
    return fig


def plot_normalized_pose_overlay(org_record, rep_record, figsize=(7, 7)):
    fig, ax = plt.subplots(figsize=figsize)
    colors = [("#1f77b4", org_record, "Original"), ("#d62728", rep_record, "Replacement")]
    for color, record, label in colors:
        point_lookup = {name: record_point(record, name, field="normalized_keypoints") for name in DOG_KEYPOINT_NAMES}
        for a_name, b_name in POSE_EDGES:
            a = point_lookup[a_name]
            b = point_lookup[b_name]
            if a is not None and b is not None:
                ax.plot([a[0], b[0]], [a[1], b[1]], color=color, linewidth=1.5, alpha=0.75)
        xs = []
        ys = []
        for point in point_lookup.values():
            if point is not None:
                xs.append(point[0])
                ys.append(point[1])
        if xs:
            ax.scatter(xs, ys, s=24, color=color, label=label)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    ax.legend()
    ax.set_title("Canonical normalized pose overlay")
    return fig


def extract_upload_entry(upload_value):
    if isinstance(upload_value, dict):
        values = list(upload_value.values())
    else:
        values = list(upload_value)
    if not values:
        return None
    return values[0]


def get_upload_name(entry):
    if entry is None:
        return None
    return entry.get("name") or entry.get("metadata", {}).get("name") or "uploaded_image.png"


def get_upload_bytes(entry):
    if entry is None:
        return None
    content = entry.get("content")
    if isinstance(content, memoryview):
        return content.tobytes()
    if isinstance(content, bytes):
        return content
    if content is None and "value" in entry:
        value = entry["value"]
        if isinstance(value, memoryview):
            return value.tobytes()
        if isinstance(value, bytes):
            return value
    return bytes(content)


def save_upload_entry(entry, dst_dir: Path, prefix: str):
    dst_dir.mkdir(parents=True, exist_ok=True)
    name = get_upload_name(entry)
    raw = get_upload_bytes(entry)
    suffix = Path(name).suffix or ".png"
    safe_name = "".join(ch if ch.isalnum() or ch in ("-", "_", ".") else "_" for ch in Path(name).stem)
    target = dst_dir / f"{prefix}_{safe_name}{suffix}"
    target.write_bytes(raw)
    return target


def fig_to_png_bytes(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    return buffer.getvalue()


def file_to_data_uri(path: Path):
    mime = "image/png"
    suffix = path.suffix.lower()
    if suffix in {".jpg", ".jpeg"}:
        mime = "image/jpeg"
    elif suffix == ".webp":
        mime = "image/webp"
    elif suffix == ".bmp":
        mime = "image/bmp"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


def image_widget_from_path(path: Path, max_width="220px"):
    return widgets.Image(
        value=path.read_bytes(),
        format=path.suffix.lower().replace(".", "") or "png",
        layout=widgets.Layout(width=max_width),
    )


def score_chip(label, value, color):
    return (
        f"<span style='display:inline-block;padding:4px 10px;border-radius:999px;"
        f"background:{color};color:white;font-size:12px;font-weight:600;margin-right:8px'>{label}: {value}</span>"
    )


def metric_bar_html(label, normalized_value, value_text, status, status_color, note):
    normalized_value = max(0.0, min(1.0, normalized_value))
    fill = int(round(100 * normalized_value))
    return f'''
    <tr>
      <td style="padding:10px 12px;font-weight:600;vertical-align:top">{label}</td>
      <td style="padding:10px 12px;min-width:260px">
        <div style="background:#ECEFF4;border-radius:999px;height:12px;overflow:hidden">
          <div style="width:{fill}%;background:{status_color};height:12px"></div>
        </div>
      </td>
      <td style="padding:10px 12px;white-space:nowrap">{value_text}</td>
      <td style="padding:10px 12px;color:{status_color};font-weight:700;white-space:nowrap">{status}</td>
      <td style="padding:10px 12px">{note}</td>
    </tr>
    '''


def render_html_table(title, rows):
    body = "".join(rows)
    html = f'''
    <div style="margin:12px 0 22px 0">
      <h3 style="margin:0 0 10px 0">{title}</h3>
      <table style="border-collapse:collapse;width:100%;font-size:14px">
        <thead>
          <tr style="background:#F6F8FA;text-align:left">
            <th style="padding:10px 12px">Metric</th>
            <th style="padding:10px 12px">Visual cue</th>
            <th style="padding:10px 12px">Value</th>
            <th style="padding:10px 12px">Assessment</th>
            <th style="padding:10px 12px">Interpretation</th>
          </tr>
        </thead>
        <tbody>{body}</tbody>
      </table>
    </div>
    '''
    return HTML(html)


def save_json(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

INDEX_RECORDS = json.loads(CURRENT_STAGE_INDEX_PATH.read_text(encoding="utf-8"))
for record in INDEX_RECORDS:
    if "shape_vector" not in record:
        record["shape_vector"] = shape_vector_from_normalized(record)
    if "shape_label" not in record:
        record["shape_label"] = shape_label_from_vector(record["shape_vector"])

POSE_MODEL = YOLO(str(POSE_MODEL_PATH))


def choose_search_pool(query_record, index_records, top_k):
    exact_pool = [r for r in index_records if r.get("current_candidate_pool_key") == query_record.get("current_candidate_pool_key")]
    same_geometry = [r for r in index_records if r.get("pose_geometry_label") == query_record.get("pose_geometry_label")]
    if len(exact_pool) >= top_k:
        return exact_pool, "exact_current_candidate_pool"
    if len(same_geometry) >= top_k:
        return same_geometry, "same_pose_geometry_label"
    return index_records, "all_current_stage_replaceable_dogs"


def score_candidate(query_record, candidate_record):
    kp_dist, shared_keypoints = keypoint_distance(query_record, candidate_record)
    vis_overlap = visibility_overlap(query_record, candidate_record)
    vis_pen = 1.0 - vis_overlap
    shape_dist, shared_shape = shape_distance(query_record, candidate_record)
    geom_pen = geometry_penalty(query_record, candidate_record)
    score = 0.72 * kp_dist + 0.12 * vis_pen + 0.16 * shape_dist + geom_pen
    return {
        "score": float(score),
        "keypoint_distance": float(kp_dist),
        "visibility_penalty": float(vis_pen),
        "visibility_overlap": float(vis_overlap),
        "shape_distance": float(shape_dist),
        "shared_keypoints": int(shared_keypoints),
        "shared_shape_features": int(shared_shape),
        "geometry_penalty": float(geom_pen),
    }


def retrieve_top_matches(query_record, index_records, top_k=50):
    search_pool, pool_strategy = choose_search_pool(query_record, index_records, top_k=top_k)
    query_resolved = str(Path(query_record["query_image_path"]).resolve())
    ranked = []
    for candidate in search_pool:
        candidate_path = ROOT / candidate["relative_path"]
        if str(candidate_path.resolve()) == query_resolved:
            continue
        metrics = score_candidate(query_record, candidate)
        ranked.append({**candidate, **metrics, "candidate_path": str(candidate_path)})
    ranked.sort(key=lambda item: item["score"])
    return ranked[:top_k], pool_strategy, len(search_pool)


def candidate_download_html(candidate_path: Path):
    uri = file_to_data_uri(candidate_path)
    filename = candidate_path.name
    return widgets.HTML(
        value=(
            f'<a download="{filename}" href="{uri}" '
            f'style="text-decoration:none">'
            f'<button style="padding:8px 12px;border:none;border-radius:10px;background:#1F6FEB;'
            f'color:white;font-weight:600;cursor:pointer">Download image</button></a>'
        )
    )


def render_query_summary(record):
    chips = [
        score_chip("View", record["view_class"], "#6F42C1"),
        score_chip("Geometry", record["pose_geometry_label"], "#0969DA"),
        score_chip("Shape", record["shape_label"], "#1A7F37"),
        score_chip("Pass", str(record["current_pipeline_pass"]), "#BC4C00" if not record["current_pipeline_pass"] else "#1A7F37"),
    ]
    html = f'''
    <div style="margin:6px 0 14px 0">
      <div style="font-size:20px;font-weight:700;margin-bottom:8px">Query analysis summary</div>
      <div style="margin-bottom:10px">{''.join(chips)}</div>
      <div style="font-size:14px;line-height:1.7">
        <b>Visible keypoints:</b> {record["visible_keypoints"]}<br/>
        <b>Visible parts:</b> {record["visible_parts_signature"]}<br/>
        <b>Current-stage reason:</b> {record["current_pipeline_reason"]}<br/>
        <b>Candidate pool key:</b> {record["current_candidate_pool_key"]}
      </div>
    </div>
    '''
    return HTML(html)


def render_candidate_cards(matches):
    cards = []
    for idx, match in enumerate(matches, start=1):
        candidate_path = Path(match["candidate_path"])
        image_box = image_widget_from_path(candidate_path, max_width="200px")
        info_html = widgets.HTML(
            value=(
                f"<div style='font-size:13px;line-height:1.55'>"
                f"<b>#{idx}</b> {candidate_path.name}<br/>"
                f"<b>Score:</b> {match['score']:.4f}<br/>"
                f"<b>Pose:</b> {match['pose_geometry_label']}<br/>"
                f"<b>Shape:</b> {match.get('shape_label', 'unknown')}<br/>"
                f"<b>Keypoint dist:</b> {match['keypoint_distance']:.4f}<br/>"
                f"<b>Visibility overlap:</b> {match['visibility_overlap']:.3f}"
                f"</div>"
            )
        )
        card = widgets.VBox(
            [image_box, info_html, candidate_download_html(candidate_path)],
            layout=widgets.Layout(border="1px solid #D0D7DE", padding="10px", width="220px", gap="8px"),
        )
        cards.append(card)
    rows = []
    for start in range(0, len(cards), 5):
        rows.append(widgets.HBox(cards[start:start + 5], layout=widgets.Layout(gap="12px", margin="0 0 14px 0")))
    return widgets.VBox(rows)


def run_query_analysis_from_path(query_path: Path, top_k=50, display_results=True):
    query_record = query_pose_record(query_path, POSE_MODEL)
    if query_record.get("status") != "ok":
        raise RuntimeError("No dog pose could be extracted from the uploaded image.")

    top_matches, pool_strategy, search_pool_size = retrieve_top_matches(query_record, INDEX_RECORDS, top_k=top_k)

    query_dir = OUTPUT_ROOT / query_path.stem
    query_dir.mkdir(parents=True, exist_ok=True)
    query_fig = draw_pose_record(query_record, image_path=query_path, figsize=(9, 7))
    query_fig_path = query_dir / "query_pose_visualization.png"
    query_fig.savefig(query_fig_path, dpi=160, bbox_inches="tight")

    copied_dir = query_dir / "top50_images"
    copied_dir.mkdir(parents=True, exist_ok=True)
    copied_manifest = []
    for rank, match in enumerate(top_matches, start=1):
        src = Path(match["candidate_path"])
        dst = copied_dir / f"{rank:02d}_{src.name}"
        shutil.copy2(src, dst)
        copied_manifest.append({"rank": rank, "source_path": str(src), "saved_path": str(dst), "score": match["score"]})

    summary = {
        "query": {k: v for k, v in query_record.items() if k not in {"keypoints_xy", "keypoints_conf", "normalized_keypoints"}},
        "query_keypoints_xy": query_record["keypoints_xy"],
        "query_keypoints_conf": query_record["keypoints_conf"],
        "query_normalized_keypoints": query_record["normalized_keypoints"],
        "pool_strategy": pool_strategy,
        "search_pool_size": search_pool_size,
        "top_k": top_k,
        "matches": [
            {
                "rank": idx,
                "relative_path": match["relative_path"],
                "candidate_path": match["candidate_path"],
                "score": match["score"],
                "pose_geometry_label": match["pose_geometry_label"],
                "shape_label": match.get("shape_label", "unknown"),
                "keypoint_distance": match["keypoint_distance"],
                "shape_distance": match["shape_distance"],
                "visibility_overlap": match["visibility_overlap"],
                "shared_keypoints": match["shared_keypoints"],
            }
            for idx, match in enumerate(top_matches, start=1)
        ],
        "saved_top50_images": copied_manifest,
    }
    save_json(query_dir / "retrieval_summary.json", summary)

    if display_results:
        display(render_query_summary(query_record))
        display(query_fig)
        plt.close(query_fig)
        display(HTML(f"<div style='margin:10px 0 16px 0;font-size:15px'><b>Pool strategy:</b> {pool_strategy} | <b>Search pool size:</b> {search_pool_size}</div>"))
        display(render_candidate_cards(top_matches))
        display(HTML(
            f"<div style='margin-top:10px;font-size:14px'>"
            f"Saved outputs to <code>{query_dir}</code></div>"
        ))
    else:
        plt.close(query_fig)
    return {"query_record": query_record, "top_matches": top_matches, "pool_strategy": pool_strategy, "search_pool_size": search_pool_size, "output_dir": str(query_dir)}


## Interactive Upload And Retrieval

Upload one dog image, click **Analyze query image**, and the notebook will display the pose visualization plus the top-50 replacement candidates directly in the notebook.


In [ ]:
status_output = widgets.Output()
result_output = widgets.Output()

query_upload = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload query image",
)
analyze_button = widgets.Button(
    description="Analyze query image",
    button_style="primary",
    icon="search",
)


def on_analyze_query(_):
    with status_output:
        clear_output(wait=True)
        print("Running query pose analysis and candidate retrieval...")
    with result_output:
        clear_output(wait=True)

    entry = extract_upload_entry(query_upload.value)
    if entry is None:
        with status_output:
            clear_output(wait=True)
            print("Please upload one dog image first.")
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    upload_dir = OUTPUT_ROOT / "_uploaded_queries"
    query_path = save_upload_entry(entry, upload_dir, f"query_{timestamp}")

    with result_output:
        run_query_analysis_from_path(query_path, top_k=TOP_K, display_results=True)

    with status_output:
        clear_output(wait=True)
        print(f"Finished. Query image saved to: {query_path}")


analyze_button.on_click(on_analyze_query)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 8px 0'>Upload one dog image</h3><div style='font-size:14px'>The notebook will estimate the pose and retrieve the 50 best replacement-dog candidates.</div>"),
    query_upload,
    analyze_button,
    status_output,
    result_output,
], layout=widgets.Layout(gap="10px")))
